<a href="https://colab.research.google.com/github/tracyhann/bsl-data-tools/blob/main/folder_navigator.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# @title 🔗 Connect to Google Drive {"run":"auto","display-mode":"form"}
# Colab cell 1: install + authenticate
!pip install -q google-api-python-client google-auth google-auth-httplib2

from google.colab import auth
auth.authenticate_user()

import google.auth
import logging
from googleapiclient.discovery import build

creds, project_id = google.auth.default()
logging.getLogger("google_auth_httplib2").setLevel(logging.ERROR)

drive_service = build("drive", "v3", credentials=creds)

In [ ]:
# @title 📁 Describe folder {"run":"auto","vertical-output":true,"display-mode":"form"}
# @markdown Enter Google Drive folder link (either copy-paste from browser directly or the shared link works!)

FOLDER_URL = "https://drive.google.com/drive/folders/1OLGHgxPg9UsBDbOH6vgSjiS6dUW0lBCF" # @param {"type":"string","placeholder":"Ex: https://drive.google.com/drive/folders/1RJiN94mftBVJ62RG2xMlUBr5_J5Gq0eA"}


from IPython.display import HTML, display
from html import escape
from typing import Optional

FOLDER_MIME = "application/vnd.google-apps.folder"
SHORTCUT_MIME = "application/vnd.google-apps.shortcut"


import re
from urllib.parse import urlparse, parse_qs
import json
import uuid


def extract_drive_id(url_or_id: str) -> str:
    """
    Extracts a Google Drive file/folder ID from:
    - raw ID
    - https://drive.google.com/drive/folders/<ID>
    - https://drive.google.com/file/d/<ID>/view
    - https://docs.google.com/document/d/<ID>/edit
    - URLs with ?id=<ID>
    """
    if not url_or_id:
        raise ValueError("Missing Google Drive URL or ID.")

    text = url_or_id.strip()

    # If it already looks like a Drive ID, return it.
    if re.fullmatch(r"[a-zA-Z0-9_-]{20,}", text):
        return text

    parsed = urlparse(text)

    # Handles URLs like: ?id=<ID>
    query_id = parse_qs(parsed.query).get("id")
    if query_id:
        return query_id[0]

    patterns = [
        r"/folders/([a-zA-Z0-9_-]+)",
        r"/file/d/([a-zA-Z0-9_-]+)",
        r"/document/d/([a-zA-Z0-9_-]+)",
        r"/spreadsheets/d/([a-zA-Z0-9_-]+)",
        r"/presentation/d/([a-zA-Z0-9_-]+)",
        r"/forms/d/([a-zA-Z0-9_-]+)",
    ]

    for pattern in patterns:
        match = re.search(pattern, text)
        if match:
            return match.group(1)

    raise ValueError(f"Could not extract a Google Drive ID from: {url_or_id}")


def get_file_metadata(file_id: str) -> dict:
    """
    Gets metadata for a Drive file/folder.
    """
    return (
        drive_service.files()
        .get(
            fileId=file_id,
            fields=(
                "id, name, mimeType, webViewLink, "
                "shortcutDetails"
            ),
            supportsAllDrives=True,
        )
        .execute()
    )


def list_children(folder_id: str) -> list[dict]:
    """
    Lists children inside a Google Drive folder.
    Includes files, folders, and shortcuts.
    """
    children = []
    page_token = None

    while True:
        response = (
            drive_service.files()
            .list(
                q=f"'{folder_id}' in parents and trashed = false",
                fields=(
                    "nextPageToken, "
                    "files(id, name, mimeType, webViewLink, shortcutDetails)"
                ),
                orderBy="folder,name",
                pageToken=page_token,
                supportsAllDrives=True,
                includeItemsFromAllDrives=True,
            )
            .execute()
        )

        children.extend(response.get("files", []))
        page_token = response.get("nextPageToken")

        if not page_token:
            break

    return children


def make_link(name: str, url: Optional[str]) -> str:
    safe_name = escape(name)

    if not url:
        return safe_name

    safe_url = escape(url, quote=True)
    return f'<a href="{safe_url}" target="_blank">{safe_name}</a>'


def build_tree_html(
    folder_id: str,
    max_depth: Optional[int] = None,
    depth: int = 0,
    visited: Optional[set[str]] = None,
    open_until_depth: int = 1,
) -> str:
    """
    Builds a clickable, collapsible HTML tree for a Google Drive folder.

    open_until_depth:
      - 0 = everything collapsed below root
      - 1 = first folder level open
      - 2 = first two folder levels open
      - etc.
    """
    if visited is None:
        visited = set()

    if folder_id in visited:
        return "<ul><li class='muted'>↩️ already visited, skipping to avoid loop</li></ul>"

    visited.add(folder_id)

    if max_depth is not None and depth > max_depth:
        return ""

    try:
        children = list_children(folder_id)
    except Exception as e:
        return f"<ul><li class='error'>❌ ERROR opening folder {escape(folder_id)}: {escape(str(e))}</li></ul>"

    if not children:
        return "<ul><li class='muted'><em>empty</em></li></ul>"

    html_parts = ["<ul class='tree-list'>"]

    for item in children:
        name = item["name"]
        mime_type = item["mimeType"]
        url = item.get("webViewLink")

        if mime_type == FOLDER_MIME:
            is_open = "open" if depth < open_until_depth else ""

            html_parts.append("<li>")
            html_parts.append(f"<details {is_open}>")
            html_parts.append(
                f"<summary>📁 {make_link(name, url)}</summary>"
            )
            html_parts.append(
                build_tree_html(
                    item["id"],
                    max_depth=max_depth,
                    depth=depth + 1,
                    visited=visited,
                    open_until_depth=open_until_depth,
                )
            )
            html_parts.append("</details>")
            html_parts.append("</li>")

        elif mime_type == SHORTCUT_MIME:
            target = item.get("shortcutDetails", {})
            target_id = target.get("targetId")
            target_mime = target.get("targetMimeType")

            # If shortcut points to folder, make it collapsible too
            if target_id and target_mime == FOLDER_MIME:
                is_open = "open" if depth < open_until_depth else ""

                html_parts.append("<li>")
                html_parts.append(f"<details {is_open}>")
                html_parts.append(
                    f"<summary>🔗 {make_link(name, url)} "
                    f"<span class='muted'>(shortcut)</span></summary>"
                )
                html_parts.append(
                    build_tree_html(
                        target_id,
                        max_depth=max_depth,
                        depth=depth + 1,
                        visited=visited,
                        open_until_depth=open_until_depth,
                    )
                )
                html_parts.append("</details>")
                html_parts.append("</li>")

            else:
                html_parts.append(
                    f"<li>🔗 {make_link(name, url)} "
                    f"<span class='muted'>(shortcut)</span></li>"
                )

        else:
            html_parts.append(
                f"<li>📄 {make_link(name, url)}</li>"
            )

    html_parts.append("</ul>")
    return "\n".join(html_parts)


def html_indent(depth: int, spaces_per_depth: int = 4) -> str:
    """
    Google Docs preserves non-breaking spaces more reliably than CSS margin.
    Primitive? yes. Effective? annoyingly yes.
    """
    return "&nbsp;" * (depth * spaces_per_depth)


def build_tree_doc_html(
    folder_id: str,
    max_depth: Optional[int] = None,
    depth: int = 0,
    visited: Optional[set[str]] = None,
) -> str:
    """
    Builds a flat, Google-Docs-friendly rich HTML block.

    No <ul>, no <details>, no <summary>.
    This avoids weird bullets/spacing when pasted into Google Docs.
    Links are preserved.
    """
    if visited is None:
        visited = set()

    if folder_id in visited:
        return (
            f"<div>{html_indent(depth)}"
            "↩️ already visited, skipping to avoid loop"
            "</div>"
        )

    visited.add(folder_id)

    if max_depth is not None and depth > max_depth:
        return ""

    try:
        children = list_children(folder_id)
    except Exception as e:
        return (
            f"<div style='color:#b91c1c;'>"
            f"{html_indent(depth)}❌ ERROR opening folder {escape(folder_id)}: {escape(str(e))}"
            "</div>"
        )

    if not children:
        return (
            f"<div style='color:#666;'>"
            f"{html_indent(depth)}<em>empty</em>"
            "</div>"
        )

    rows = []

    for item in children:
        name = item["name"]
        mime_type = item["mimeType"]
        url = item.get("webViewLink")
        indent = html_indent(depth)

        if mime_type == FOLDER_MIME:
            rows.append(
                f"<div>{indent}📁 {make_link(name, url)}</div>"
            )

            rows.append(
                build_tree_doc_html(
                    item["id"],
                    max_depth=max_depth,
                    depth=depth + 1,
                    visited=visited,
                )
            )

        elif mime_type == SHORTCUT_MIME:
            target = item.get("shortcutDetails", {})
            target_id = target.get("targetId")
            target_mime = target.get("targetMimeType")

            rows.append(
                f"<div>{indent}🔗 {make_link(name, url)} "
                f"<span style='color:#777;'>(shortcut)</span></div>"
            )

            if target_id and target_mime == FOLDER_MIME:
                rows.append(
                    build_tree_doc_html(
                        target_id,
                        max_depth=max_depth,
                        depth=depth + 1,
                        visited=visited,
                    )
                )

        else:
            rows.append(
                f"<div>{indent}📄 {make_link(name, url)}</div>"
            )

    return "\n".join(rows)


def build_tree_text(
    folder_id: str,
    max_depth: int | None = None,
    depth: int = 0,
    visited: set[str] | None = None,
) -> str:
    """
    Builds a plain-text directory tree that can be copied.
    """
    if visited is None:
        visited = set()

    if folder_id in visited:
        return "  " * depth + "↩️ already visited, skipping to avoid loop\n"

    visited.add(folder_id)

    if max_depth is not None and depth > max_depth:
        return ""

    try:
        children = list_children(folder_id)
    except Exception as e:
        return "  " * depth + f"❌ ERROR opening folder {folder_id}: {e}\n"

    if not children:
        return "  " * depth + "empty\n"

    lines = []

    for item in children:
        name = item["name"]
        mime_type = item["mimeType"]

        if mime_type == FOLDER_MIME:
            lines.append("  " * depth + f"📁 {name}/")
            lines.append(
                build_tree_text(
                    item["id"],
                    max_depth=max_depth,
                    depth=depth + 1,
                    visited=visited,
                ).rstrip("\n")
            )

        elif mime_type == SHORTCUT_MIME:
            target = item.get("shortcutDetails", {})
            target_id = target.get("targetId")
            target_mime = target.get("targetMimeType")

            lines.append("  " * depth + f"🔗 {name} (shortcut)")

            if target_id and target_mime == FOLDER_MIME:
                lines.append(
                    build_tree_text(
                        target_id,
                        max_depth=max_depth,
                        depth=depth + 1,
                        visited=visited,
                    ).rstrip("\n")
                )

        else:
            lines.append("  " * depth + f"📄 {name}")

    return "\n".join(line for line in lines if line.strip()) + "\n"


def make_copyable_directory_html(
    root_name: str,
    tree_html: str,
    tree_text: str,
    root_url: Optional[str] = None,
    copy_body_html: Optional[str] = None,
    button_color: str = "#476A70",
    button_hover_color: str = "#6222A3",
    button_text_color: str = "white",
) -> str:
    safe_root = escape(root_name)
    safe_tree_text = escape(tree_text)
    root_label_html = make_link(root_name, root_url) if root_url else safe_root

    if copy_body_html is None:
        copy_body_html = tree_html

    # This is the actual rich HTML block copied to clipboard.
    # Google Docs should render this as formatted text with clickable links.
    copy_html = f"""
<div style="font-family: Arial, sans-serif; font-size: 11pt; line-height: 1.35;">
  <div style="font-weight: 700; margin-bottom: 6px;">📁 {root_label_html}</div>
  {copy_body_html}
</div>
"""

    uid = "copy_dir_" + uuid.uuid4().hex[:10]

    js_copy_html = json.dumps(copy_html)
    js_tree_text = json.dumps(tree_text)

    return f"""
<div style="font-family: Arial, sans-serif; line-height: 1.55;">

  <style>
    .copy-dir-btn {{
      background: {button_color};
      color: {button_text_color};
      border: none;
      border-radius: 8px;
      padding: 10px 14px;
      font-size: 18px;
      font-weight: 700;
      cursor: pointer;
      margin: 8px 8px 12px 0;
    }}

    .copy-dir-btn:hover {{
      background: {button_hover_color};
    }}

    .copy-dir-status {{
      color: #8ab4f8;
      font-size: 17px;
      margin-left: 8px;
    }}

    .dir-copy-box {{
      width: 100%;
      height: 180px;
      font-family: monospace;
      font-size: 16px;
      white-space: pre;
      margin-top: 8px;
      display: none;
    }}
  </style>

  <h3>📁 {root_label_html}</h3>

  <button id="{uid}_btn" class="copy-dir-btn">
    📋 Copy this tree 🌴
  </button>

  <span id="{uid}_status" class="copy-dir-status"></span>

  <textarea id="{uid}_fallback" class="dir-copy-box">{safe_tree_text}</textarea>

  <script>
    (() => {{
      const html = {js_copy_html};
      const text = {js_tree_text};

      const btn = document.getElementById("{uid}_btn");
      const status = document.getElementById("{uid}_status");
      const fallbackBox = document.getElementById("{uid}_fallback");

      async function copyRichHtml() {{
        try {{
          if (navigator.clipboard && window.ClipboardItem) {{
            const item = new ClipboardItem({{
              "text/html": new Blob([html], {{type: "text/html"}}),
              "text/plain": new Blob([text], {{type: "text/plain"}})
            }});

            await navigator.clipboard.write([item]);
            status.textContent = "Copied this tree with embedded links. Feel free to paste into Docs directly!";
            return;
          }}

          throw new Error("Rich clipboard unavailable.");
        }} catch (err) {{
          try {{
            const holder = document.createElement("div");
            holder.innerHTML = html;
            holder.contentEditable = "true";
            holder.style.position = "fixed";
            holder.style.left = "-999999px";
            holder.style.top = "0";
            holder.style.whiteSpace = "normal";

            document.body.appendChild(holder);

            const range = document.createRange();
            range.selectNodeContents(holder);

            const selection = window.getSelection();
            selection.removeAllRanges();
            selection.addRange(range);

            const ok = document.execCommand("copy");

            selection.removeAllRanges();
            document.body.removeChild(holder);

            if (ok) {{
              status.textContent = "Copied rich linked block. Paste into Google Docs.";
              return;
            }}

            throw new Error("Fallback rich copy failed.");
          }} catch (fallbackErr) {{
            fallbackBox.style.display = "block";
            fallbackBox.value = text;
            fallbackBox.focus();
            fallbackBox.select();
            status.textContent = "Rich copy blocked. Plain text selected below.";
          }}
        }}
      }}

      btn.addEventListener("click", copyRichHtml);
    }})();
  </script>

  {tree_html}
</div>
"""


root_folder_id = extract_drive_id(FOLDER_URL)
root_meta = get_file_metadata(root_folder_id)

tree_html = build_tree_html(root_folder_id)
tree_text = f'{root_meta["name"]}/\n' + build_tree_text(root_folder_id)

copy_body_html = build_tree_doc_html(root_folder_id)

root_html = make_copyable_directory_html(
    root_name=root_meta["name"],
    tree_html=tree_html,
    tree_text=tree_text,
    root_url=root_meta.get("webViewLink"),
    copy_body_html=copy_body_html,
)

display(HTML(root_html))